# Specialiserede Modeller — 3: CNN — netværk der kan SE

Et almindeligt netværk ser et billede som 784 tal i én lang række og aner ikke, at pixels
har naboer. I dag gør vi noget ved det: **konvolutionelle neurale netværk (CNN'er)** er
bygget PRÆCIS til at udnytte, at billeder har struktur — og de er grunden til, at
computere i dag kan se.

Dagens data: **FashionMNIST** — 28×28-billeder af tøj i 10 kategorier. Billederne er små
gråtonebilleder, og tøjtyperne ligner ofte hinanden meget (prøv selv at skelne en skjorte
fra en frakke i 28×28...).

> **Om opgaverne:** Noget af det her er nyt og kan føles udfordrende i starten — og det er helt okay. Vi forklarer hvert skridt så klart og tydeligt, vi kan, og der er et hint til hver opgave, hvis du går i stå. Tag dig endelig god tid.
>
> Notebooken er selvkørende — du kan tage emnets notebooks i den rækkefølge, du vil. Der
> er med vilje flere opgaver, end du kan nå: nederst finder du en sektion med **Ekstra
> opgaver**, hvis du når hele vejen igennem og har lyst til mere, og opgaver mærket
> **(find fejlen)** indeholder en bevidst fejl, som du skal finde og rette.

## Setup

In [ ]:
!pip install -q kagglehub

# Plottehjælpere + FashionMNIST fra GitHub (Plan B: upload filerne via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/fashion_traen_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/fashion_test_lille.csv.gz

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from helpers import show_images, plot_confusion_matrix

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
train_df = pd.read_csv("fashion_traen_lille.csv.gz")
test_df = pd.read_csv("fashion_test_lille.csv.gz")
print("træning:", train_df.shape, "| test:", test_df.shape)

> **Plan B:** Hvis Kaggle driller, så fjern `#`'erne nedenfor og kør — filerne fra
> vores GitHub er allerede skåret ned, så spring i så fald `sample`-linjerne ovenfor over.

Formatet er som MNIST: første kolonne er `label`, resten er 784
pixels. Men nu er der en VIGTIG ny detalje: et CNN vil have billederne som **billeder**,
ikke som flade rækker. Shapen skal være `(antal, kanaler, højde, bredde)` — kanaler er
1 for gråtoner (og 3 for RGB-farver, som du møder i en af ekstra-opgaverne til sidst):

In [ ]:
class_names = ["T-shirt", "bukser", "sweater", "kjole", "frakke",
               "sandal", "skjorte", "sneaker", "taske", "støvle"]

X_train = torch.tensor(train_df.drop(columns=["label"]).values / 255.0,
                       dtype=torch.float32).reshape(-1, 1, 28, 28)
y_train = torch.tensor(train_df["label"].values, dtype=torch.long)
X_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0,
                      dtype=torch.float32).reshape(-1, 1, 28, 28)
y_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print("X_traen:", X_train.shape, "  ← (antal, kanaler, højde, bredde)")
show_images(X_train, y_train, n=10, class_names=class_names)

# 1: Konvolution — den glidende linse

## Hvorfor ikke et almindeligt dense-netværk?

I MNIST-notebooken fladede vi billedet ud til 784 tal, og hver eneste pixel fik sin
egen vægt. To problemer:

1. **Naboskab smides væk**: for netværket er pixel 5 og pixel 33 to løsrevne tal ved
 siden af hinanden i den lange række — det har ingen måde at vide, at de i virkeligheden
 sidder lige oven over hinanden i billedet.
2. **Intet genbrug**: netværket lærer at genkende en snude i øverste venstre hjørne —
 og aner så INTET, når snuden optræder nederst til højre. Alt skal læres forfra for
 hver position.

## Idéen: en lille kerne, der glider

En **konvolution** løser begge dele. Tag en lille matrix — en **kerne** (fx 2×2 eller
3×3) — og lad den GLIDE hen over billedet. På hver position ganges kernen med det
stykke af billedet, den ligger oven på, og summen skrives i output. Samme kerne,
alle positioner:

In [ ]:
image = torch.tensor([[1., 2., 0., 1.],
                        [0., 1., 3., 1.],
                        [2., 1., 0., 0.],
                        [1., 3., 1., 2.]])

kerne = torch.tensor([[1., 0.],
                      [0., -1.]])

# F.conv2d vil have shapen (antal, kanaler, højde, bredde) — derfor reshape:
result = F.conv2d(image.reshape(1, 1, 4, 4), kerne.reshape(1, 1, 2, 2))
print(result.squeeze())

Tjek det øverste venstre tal i hånden: kernen ligger på $\begin{pmatrix}1 & 2\\0 & 1\end{pmatrix}$,
og $1\cdot 1 + 2\cdot 0 + 0\cdot 0 + 1\cdot(-1) = 0$. Kernen `[[1,0],[0,-1]]` beregner
altså "pixel minus dens diagonale nabo" — den reagerer på ÆNDRINGER på skrå.

Bemærk output-størrelsen: et 4×4-billede og en 2×2-kerne giver 3×3 output — kernen kan
stå på 3×3 forskellige positioner. Generelt: **output = billede − kerne + 1** (uden
padding og stride, mere om dem om lidt).

## Kerner kan SE ting — fx kanter

Her kommer det fede: forskellige kerner opdager forskellige mønstre. Den her berømte
kerne (Sobel-kernen) reagerer kraftigt på **lodrette kanter** — steder hvor billedet
skifter fra lyst til mørkt i vandret retning:

In [ ]:
vertical_edge = torch.tensor([[1., 0., -1.],
                            [2., 0., -2.],
                            [1., 0., -1.]])

shoe = X_train[y_train == 7][0]        # en sneaker fra datasættet

edges = F.conv2d(shoe.reshape(1, 1, 28, 28), vertical_edge.reshape(1, 1, 3, 3))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(shoe.squeeze(), cmap="gray")
axes[0].set_title("original")
axes[1].imshow(edges.squeeze(), cmap="gray")
axes[1].set_title("lodret-kant-kerne")
for axis in axes:
    axis.axis("off")
plt.show()

Kernen har fremhævet skoens lodrette kanter og ignoreret resten! I et CNN er det
PRÆCIS sådan, de første lag ser: små kerner, der finder kanter, streger og pletter. Den
eneste forskel er, at CNN'et **lærer kernernes tal selv** med gradient descent — i
stedet for at få dem foræret af os.

## Padding, stride og pooling

Tre små begreber, så kan I læse enhver CNN-arkitektur:

- **Padding**: læg en ramme af nuller om billedet, så kernen også kan stå på kanten —
 `padding=1` med en 3×3-kerne bevarer billedstørrelsen.
- **Stride**: lad kernen hoppe flere pixels ad gangen — `stride=2` halverer outputtet.
- **Max-pooling**: skru ned for opløsningen ved at tage MAKSIMUM i små vinduer —
 `MaxPool2d(2)` deler billedet i 2×2-felter og beholder ét tal pr. felt. Det gør
 netværket mindre pixel-pernittengrynet ("der var en kant CIRKA her").

Formlen for output-størrelsen: $\text{ud} = \lfloor(\text{ind} - \text{kerne} + 2\cdot\text{padding})/\text{stride}\rfloor + 1$

In [ ]:
number = torch.tensor([[1., 3., 2., 4.],
                    [5., 2., 1., 0.],
                    [0., 1., 6., 2.],
                    [3., 2., 1., 1.]])

print(F.max_pool2d(number.reshape(1, 1, 4, 4), kernel_size=2).squeeze())

### Opgaver

##### Opgave 1.1
Vi tjekker konvolutionen efter i hånden, ét tal ad gangen.

Prøv at regne det MIDTERSTE tal i 3×3-outputtet fra det første eksempel ovenfor
(4×4-billedet og kernen `[[1,0],[0,-1]]`). Kernen lægges, så dens øverste venstre hjørne
står på tallet 1 i midten (position række 1, kolonne 1). Læg de fire produkter sammen, og
sammenlign med `result` ovenfor.

Kig på cellen ovenfor: det øverste venstre tal blev regnet på præcis samme måde — nu
flytter du kernen ét skridt ind mod midten.

Hint: kernen `[[1,0],[0,-1]]` betyder "tallet kernen ligger på, minus tallet skråt nedad
til højre". Hvilke to tal lander de to hjørner på?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.2
Nu prøver du en hel lille konvolution, hvor du kan overskue alle tallene på én gang.

Billedet og kernen står klar i cellen nedenfor (3×3-billede, 2×2-kerne → 2×2 output). Se
om du kan regne alle fire output-tal i hånden først — skriv dine gæt i kommentarlinjerne
— og tjek så dig selv med `F.conv2d`.

Husk mønstret fra opgave 1.1: for hvert output-tal lægger du kernen et sted, ganger
overlap-for-overlap og lægger sammen. Kernen kan stå på 2×2 forskellige steder.

Hint: øverst venstre lægger kernen sig på billedets øverste venstre hjørne, altså
$2\cdot 1 + 0\cdot(-1) + 1\cdot 0 + 3\cdot 1$. Flyt derefter kernen ét skridt til højre og gentag.

In [ ]:
mit_image = torch.tensor([[2., 0., 1.],
                            [1., 3., 0.],
                            [0., 2., 2.]])
min_kerne = torch.tensor([[1., -1.],
                          [0., 1.]])

# Regn de fire tal i hånden FØRST — skriv dine gæt her:
# øverst venstre: ...   øverst højre: ...
# nederst venstre: ...  nederst højre: ...

print(F.conv2d(mit_image.reshape(1, 1, 3, 3), min_kerne.reshape(1, 1, 2, 2)).squeeze())

##### Opgave 1.3
Vi ser på, at to forskellige kerner kan gøre vidt forskellige ting ved det samme billede.

Nedenfor ligger to kerner: `identity` og `smear`. Prøv at køre cellen med hver af dem på
sneakeren (skift `kerne_test = identity` ud med `smear`), og læg mærke til, hvad der sker
med billedet. Prøv bagefter at sætte ord på, hvorfor tallene i kernen giver netop det
resultat.

Kig på tallene i cellen: `identity` har et 1-tal i midten og nuller udenom, mens `smear`
er lutter 1/9 — som et gennemsnit af et lille område.

Hint: hvad sker der med en pixel, hvis den ganges med 1, og alle otte naboer ganges med
0? Og hvad betyder det at erstatte hver pixel med gennemsnittet af sig selv og sine naboer?

In [ ]:
identity = torch.tensor([[0., 0., 0.],
                          [0., 1., 0.],
                          [0., 0., 0.]])

smear = torch.ones(3, 3) / 9     # alle tal er 1/9

kerne_test = identity    # ← prøv begge
ud = F.conv2d(shoe.reshape(1, 1, 28, 28), kerne_test.reshape(1, 1, 3, 3))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(shoe.squeeze(), cmap="gray")
axes[1].imshow(ud.squeeze(), cmap="gray")
for axis in axes:
    axis.axis("off")
plt.show()

##### Opgave 1.4
Sobel-kernen fandt LODRETTE kanter. Nu bygger vi dens søster, der finder VANDRETTE kanter.

En vandret-kant-kerne er den lodrette kerne vippet 90 grader. I cellen nedenfor er otte af
de ni tal fyldt ud for dig — se om du kan udfylde det sidste, så kernen er færdig. Kør den
så på sneakeren, og prøv den bagefter på en taske (`X_train[y_train == 8][0]`), og
sammenlign de to kant-billeder.

Kig på `vertical_edge` i Sobel-cellen ovenfor: den vandrette udgave bruger de samme tal, men
drejet — de kraftige tal ligger nu øverst og nederst i stedet for i siderne.

Hint: øverste og nederste række skal spejle hinanden. Øverste række er `[1, 2, 1]` — hvad
må den nederste så være?

In [ ]:
# vertical_edge fandt LODRETTE kanter. Vend den 90 grader, så finder den VANDRETTE.
# Otte af de ni tal er fyldt ud — mangler kun ét i nederste højre hjørne:
horizontal_edge = torch.tensor([[ 1.,  2.,  1.],
                                [ 0.,  0.,  0.],
                                [-1., -2., ...]])   # <-- udfyld sidste tal (nederste række skal spejle den øverste)

ud = F.conv2d(shoe.reshape(1, 1, 28, 28), horizontal_edge.reshape(1, 1, 3, 3))
plt.imshow(ud.squeeze(), cmap="gray")
plt.axis("off")
plt.show()

# prøv den bagefter på en taske: skift shoe ud med X_train[y_train == 8][0]

##### Opgave 1.5 (find fejlen)
Her øver vi os i at læse PyTorch's shape-fejl — dem møder du tit, når du arbejder med
billeder.

Koden nedenfor vil køre en kerne hen over sneakeren, men PyTorch protesterer over
dimensionerne. Prøv at køre cellen, læs fejlbeskeden, og se om du kan rette den.

Husk fra afsnittet ovenfor: `F.conv2d` vil have billedet i formen
`(antal, kanaler, højde, bredde)` — altså fire tal. Kig på, hvad `shoe.squeeze()` giver.

Hint: `shoe` frisk fra datasættet er allerede et billede — men `.squeeze()` maser
dimensionerne væk. Hvad ville `.reshape(1, 1, 28, 28)` give i stedet?

In [ ]:
kerne = torch.tensor([[1., -1.]])
ud = F.conv2d(shoe.squeeze(), kerne)
print(ud.shape)

##### Opgave 1.6
Vi ser på, hvordan padding ændrer størrelsen af det, konvolutionen spytter ud.

Cellen kører allerede 3×3-kernen på sneakeren med `padding=0`. Prøv at ændre tallet til
`padding=1` og derefter `padding=5`, og tjek output-shapen hver gang. Kig især på billedet
ved `padding=5` — hvad er det for en sort ramme, der dukker op?

Husk fra afsnittet ovenfor: padding lægger en ramme af nuller uden om billedet, før kernen
glider hen over det.

Hint: nuller tegnes som sort. Hvor bred bliver rammen, når du polstrer med 5 rækker nuller
hele vejen rundt?

In [ ]:
ud = F.conv2d(shoe.reshape(1, 1, 28, 28), vertical_edge.reshape(1, 1, 3, 3), padding=0)   # <-- prøv 1 og 5
print(ud.shape)
plt.imshow(ud.squeeze(), cmap="gray")
plt.axis("off")
plt.show()

##### Opgave 1.7
Nedenfor tjekker vi størrelses-formlen fra afsnittet ovenfor mod virkeligheden.

Prøv at oversætte formlen
$\text{ud} = \lfloor(\text{ind} - \text{kerne} + 2\cdot\text{padding})/\text{stride}\rfloor + 1$
til Python inde i `output_size`. Loopet nedenunder kører den mod PyTorch i tre tilfælde —
hvis din formel er rigtig, står de to tal ens hver gang.

Husk fra afsnittet ovenfor: kernen kan kun stå de steder, hvor den er helt inde over
billedet (plus eventuel padding), og stride bestemmer, hvor store spring den tager.

Hint: brug heltalsdivision `//` i stedet for `/` — så runder Python automatisk ned, ligesom
gulv-tegnet $\lfloor\ \rfloor$ i formlen.

In [ ]:
def output_size(ind, kerne, padding, stride):
    return ...   # <-- oversæt formlen: (ind - kerne + 2*padding) // stride + 1

for kerne_str, padding, stride in [(3, 0, 1), (3, 1, 2), (5, 2, 2)]:
    beregnet = output_size(28, kerne_str, padding, stride)
    faktisk = F.conv2d(shoe.reshape(1, 1, 28, 28),
                       torch.ones(1, 1, kerne_str, kerne_str),
                       padding=padding, stride=stride).shape[-1]
    print(f"kerne {kerne_str}, padding {padding}, stride {stride}: formel {beregnet}, PyTorch {faktisk}")

##### Opgave 1.8
Vi ser på, hvad max-pooling gør, når man bruger det flere gange i træk.

Cellen kører allerede én max-pooling på kant-billedet fra Sobel-eksemplet. Se om du kan
tilføje ÉN pooling mere — denne gang på resultatet af den første — så du ender med tre
billeder at sammenligne. Kig så på de tre: hvad overlever poolingen, og hvad forsvinder?

Husk fra afsnittet ovenfor: max-pooling deler billedet i 2×2-felter og beholder kun det
STØRSTE tal i hvert felt — så opløsningen halveres for hver gang.

Hint: du poolede `edges` til `pooled`. Gør præcis det samme igen, men med `pooled` som
input i stedet for `edges`.

In [ ]:
edges = F.conv2d(shoe.reshape(1, 1, 28, 28), vertical_edge.reshape(1, 1, 3, 3))
pooled = F.max_pool2d(edges, kernel_size=2)
pooled2 = ...   # <-- pool pooled ÉN gang til (samme kald: F.max_pool2d(pooled, kernel_size=2))

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(edges.squeeze(), cmap="gray")
axes[0].set_title(f"kanter {tuple(edges.squeeze().shape)}")
axes[1].imshow(pooled.squeeze(), cmap="gray")
axes[1].set_title(f"poolet {tuple(pooled.squeeze().shape)}")
axes[2].imshow(pooled2.squeeze(), cmap="gray")
axes[2].set_title(f"poolet ×2 {tuple(pooled2.squeeze().shape)}")
for axis in axes:
    axis.axis("off")
plt.show()

##### Opgave 1.9
Vi sammenligner de to måder at bruge vægte på: dense-laget mod konvolutionslaget.

Dense-laget lærer én vægt pr. pixel pr. neuron. Konvolutionslaget genbruger de samme 9 tal
(3×3-kernen) over HELE billedet. Prøv at finde to grunde til, at det genbrug er smart —
tænk på (1) antal parametre og (2) hvad der sker, når motivet flytter sig i billedet.

Husk sneakeren fra opgave 1.4: den samme kant-kerne fandt kanter, uanset hvor på billedet
kanten lå.

Hint: hvor mange tal skal netværket lære, hvis ét sæt på 9 dækker hele billedet — i stedet
for én vægt pr. af de 784 pixels? Og skal en snude læres forfra, hvis den flytter sig?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

# 2: Byg et CNN — og lad det lære kernerne selv

Nu samler vi klodserne. Et klassisk lille CNN ser sådan ud:

```
billede (1×28×28)
  → Conv2d(1→16 kerner, 3×3, padding=1) → ReLU → MaxPool(2)     "find 16 slags småmønstre, zoom ud"
  → Conv2d(16→32 kerner, 3×3, padding=1) → ReLU → MaxPool(2)    "kombinér til 32 større mønstre, zoom ud"
  → flad ud → Linear(32·7·7 → 10)                               "afgør: hvilken tøjtype?"
```

Bemærk logikken: konvolutionslagene finder mønstre (kanter → hjørner → ærmer → sko-form),
poolingen zoomer gradvist ud, og til allersidst samler ét dense-lag det hele til en
beslutning. Størrelserne undervejs: 28 → 28 → 14 → 14 → 7 (padding=1 bevarer størrelsen,
poolingen halverer — regn selv efter med formlen fra 1.7!).

Og træningsloopet? **Et helt almindeligt træningsloop** — CrossEntropyLoss, Adam,
mini-batches. Et CNN er et helt almindeligt `nn.Module` — forskellen er kun, hvilke lag der sidder indeni:

In [ ]:
class ClothesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)    # 1 kanal ind → 16 kerner
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)   # 16 ind → 32 kerner
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(32 * 7 * 7, 10)              # 32 kort á 7×7 → 10 klasser

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))    # 28 → 28 → 14
        x = self.pool(self.activation(self.conv2(x)))    # 14 → 14 → 7
        x = x.reshape(x.shape[0], -1)                    # flad ud: (N, 1568)
        return self.fc(x)                                # rå point (CrossEntropy-reglen!)

model = ClothesNet()
print(model)
print("parametre:", sum(p.numel() for p in model.parameters()))

In [ ]:
import time

torch.manual_seed(42)
model = ClothesNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Det her er et helt almindeligt træningsloop: for hver epoke løber vi hele
# datasættet igennem i portioner på 64 billeder, regner tabet, og lader optimizeren
# justere vægtene et lille skridt. Intet nyt at lære her — kun lagene indeni modellen er nye.
start = time.time()
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(model(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()
    print(f"epoke {epoch + 1}: tab på sidste portion = {loss.item():.4f}")
print(f"træningstid: {time.time() - start:.0f} s")

In [ ]:
with torch.no_grad():
    pred = model(X_test).argmax(dim=1)
print(f"CNN test-accuracy: {(pred == y_test).float().mean().item():.1%}")

## Duellen: CNN mod dense-netværk

Er al konvolutions-besværet det værd? Lad os træne dense-nettet fra
Intro-ML på præcis samme data — og sammenligne både accuracy OG antal parametre:

In [ ]:
torch.manual_seed(42)
dense_model = nn.Sequential(nn.Flatten(),                 # (N,1,28,28) → (N,784)
                            nn.Linear(784, 128), nn.ReLU(),
                            nn.Linear(128, 10))
optimizer = torch.optim.Adam(dense_model.parameters(), lr=0.001)

for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(dense_model(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    dense_pred = dense_model(X_test).argmax(dim=1)

print(f"CNN:   {(pred == y_test).float().mean().item():.1%}  med {sum(p.numel() for p in model.parameters()):>7} parametre")
print(f"dense: {(dense_pred == y_test).float().mean().item():.1%}  med {sum(p.numel() for p in dense_model.parameters()):>7} parametre")

CNN'et vinder — med ca. **5× færre parametre**. Struktur-viden slår rå størrelse.
(Og forspringet vokser med mere data og træning: på det fulde FashionMNIST når denne
arkitektur ~90 %+, og på rigtige fotos er dense-net helt chanceløse.)

## Hvor tager modellen fejl? Forvirringsmatricen

Accuracy er ét tal — **forvirringsmatricen** viser HVILKE klasser der forveksles:

In [ ]:
plot_confusion_matrix(y_test, pred, class_names=class_names)

Kig på de store tal UDEN FOR diagonalen: skjorte ↔ T-shirt ↔ frakke ↔ sweater
forveksles flittigt (fair nok — i 28×28 gråtoner!), mens bukser og tasker næsten aldrig
rammes forkert.

## Kig ind i maskinen: de lærte kerner

Vi håndbyggede kant-kerner i afsnit 1 — hvad har NETVÆRKET selv fundet på? `conv1` har
16 kerner à 3×3, og vi kan også se **feature maps**: hvad hver kerne "ser" i et
konkret billede:

In [ ]:
kerner = model.conv1.weight.detach()          # shape (16, 1, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, axis in enumerate(axes.ravel()):
    axis.imshow(kerner[i, 0], cmap="gray")
    axis.set_title(f"kerne {i}", fontsize=8)
    axis.axis("off")
plt.suptitle("De 16 lærte kerner i conv1")
plt.show()

In [ ]:
with torch.no_grad():
    feature_maps = model.activation(model.conv1(shoe.reshape(1, 1, 28, 28)))

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, axis in enumerate(axes.ravel()):
    axis.imshow(feature_maps[0, i], cmap="gray")
    axis.axis("off")
plt.suptitle("Hvad de 16 kerner ser i sneakeren (feature maps)")
plt.show()

### Opgaver

##### Opgave 2.1
Vi kigger på de billeder, hvor CNN'et tager fejl — det siger tit mere end selve accuracy'en.

Cellen nedenfor bruger `show_images` til at vise 10 af de billeder, netværket gætter
FORKERT (mønstret er `(pred != y_test).nonzero()`). Kør den, og se om
fejlene passer med forvirringsmatricens værste par. Er der nogen, hvor du selv ville have
gættet det samme som modellen?

Kig på forvirringsmatricen ovenfor: de store tal uden for diagonalen var netop de
tøjtyper, der ligner hinanden mest i 28×28 gråtoner.

Hint: hver billedtekst viser både det rigtige svar og modellens gæt — læg mærke til, om de
forvekslede par er de samme, som matricen udpegede (skjorte, T-shirt, frakke, sweater).

In [ ]:
forkerte = (pred != y_test).nonzero().squeeze()
print("antal fejl:", len(forkerte))

show_images(X_test[forkerte[:10]], y_test[forkerte[:10]],
             predictions=pred[forkerte[:10]], n=10, class_names=class_names)

##### Opgave 2.2
Vi ser på, om et bredere netværk — flere kerner pr. lag — gætter bedre.

Cellen nedenfor er en kopi af `ClothesNet`, men bygget så du kan skrue op. Prøv at ændre
kerne-antallet fra (16, 32) til (32, 64) de tre steder, pilene peger, og kør så. Hvad sker
der med accuracy og træningstid?

Husk togskinne-reglen: hvert lags input skal matche det forrige lags output —
laver du conv2 om til 64 kerner ud, skal `fc`-lagets input følge med.

Hint: `fc`-laget regner med `32 * 7 * 7`, fordi conv2 gav 32 kort. Hvis conv2 nu giver 64
kort, hvad skal `32` så laves om til?

In [ ]:
class BigClothesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)    # <-- ret 16 til 32
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)   # <-- ret 16, 32 til 32, 64
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(32 * 7 * 7, 10)              # <-- ret 32 til 64 (følg conv2's output)

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
stort = BigClothesNet()
optimizer = torch.optim.Adam(stort.parameters(), lr=0.001)
start = time.time()
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(stort(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()
with torch.no_grad():
    acc = (stort(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy {acc.item():.1%} på {time.time() - start:.0f} s")

##### Opgave 2.3
Vi tæller efter, hvor CNN'ets sparsommelighed egentlig kommer fra.

Cellen tæller allerede CNN'ets parametre i `cnn_total`. Se om du kan udfylde `dense_total`
på samme måde og få forholdet mellem de to modeller frem. Loopet nederst bryder CNN'ets
parametre op pr. lag — kig så på, hvor langt de FLESTE af dem sidder: i konvolutionslagene
eller i `fc`-laget?

Kig på linjen ovenover: `cnn_total` tælles med præcis det samme mønster, nu over
`model.parameters()`.

Hint: dense-nettet hedder `dense_model`. Byt `model` ud med det i optællingen.

In [ ]:
cnn_total = sum(p.numel() for p in model.parameters())
dense_total = ...   # <-- tæl dense_model's parametre (samme mønster som cnn_total ovenfor)

print(f"CNN: {cnn_total}, dense: {dense_total}, forhold: {dense_total / cnn_total:.1f}x")

# og fordelt på lag i CNN'et:
for name, p in model.named_parameters():
    print(f"{name:15s} {p.numel():>6} tal")

##### Opgave 2.4 (find fejlen)
Her finder du en fejl, der handler om formen på data mellem lagene.

Nogen har slettet en linje i `forward`, og nu kaster netværket en lang shape-fejl, når det
kaldes. Prøv at køre cellen, læs fejlen, og se om du kan sætte den manglende linje ind igen.

Kig på `ClothesNet` længere oppe: dens `forward` gjorde noget ved data lige inden `fc` —
noget conv-lagene efterlader, som et dense-lag ikke kan spise direkte.

Hint: `fc`-laget vil have en flad række tal pr. billede, men conv-lagene giver en 3D-stak
af feature maps. Hvad var det for en `reshape`-linje, der fladede dem ud?

In [ ]:
class BrokenNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        return self.fc(x)

model_o = BrokenNet()
print(model_o(X_test[:4]))

##### Opgave 2.5
Vi gør netværket dybere med endnu et konvolutionslag.

Cellen har allerede fået et TREDJE lag `conv3` (32 → 64 kerner, UDEN padding, ingen pooling
efter — 7×7 er allerede småt). Se om du kan udfylde `fc`-lagets input, så det passer med
det, `conv3` giver ud. Brug formlen fra opgave 1.7 til at regne størrelsen ud FØR du kører.
Blev accuracy bedre?

Husk fra 1.7: en 3×3-kerne uden padding gør et 7×7-kort en smule mindre — regn selv efter,
hvor stort det bliver.

Hint: `fc`-input = antal kerner × højde × bredde. Conv3 giver 64 kerner, og 7×7 bliver til
5×5 uden padding (7 − 3 + 1). Hvad giver 64 · 5 · 5?

In [ ]:
class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3)     # ingen padding!
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(..., 10)          # <-- conv3 (3×3, ingen padding) krymper 7×7-kortene — brug 1.7-formlen (64 kerner ud)

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        x = self.activation(self.conv3(x))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
deep = DeepNet()
optimizer = torch.optim.Adam(deep.parameters(), lr=0.001)
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(deep(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()
with torch.no_grad():
    acc = (deep(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%}")

##### Opgave 2.6
Nu tester vi den robusthed, du tænkte over i opgave 1.9: hvad sker der, når motivet flytter sig?

Cellen forskyder ALLE testbilleder 3 pixels til højre med `torch.roll`. Se om du kan
udfylde dense-nettets accuracy på de forskudte billeder, så du kan sammenligne de to
modeller. Hvem tåler flytningen bedst — og hvorfor er ingen af dem helt uberørt?

Kig på linjen ovenover: `cnn_acc` regnes på præcis den måde, du skal bruge til `dense_acc`
— nu med den anden model.

Hint: CNN'ets kerner glider hen over hele billedet, så de finder samme mønster, uanset hvor
det ligger. Dense-nettet har én fast vægt pr. pixel-position — hvad sker der så, når alt
rykker 3 pixels?

In [ ]:
X_test_shifted = torch.roll(X_test, shifts=3, dims=3)   # 3 pixels mod højre

show_images(X_test_shifted, y_test, n=5, class_names=class_names)

with torch.no_grad():
    cnn_acc = (model(X_test_shifted).argmax(dim=1) == y_test).float().mean()
    dense_acc = ...   # <-- samme udregning som cnn_acc, men med dense_model i stedet for model
print(f"forskudt: CNN {cnn_acc.item():.1%} | dense {dense_acc.item():.1%}")

##### Opgave 2.7
Vi finder det billede, modellen er allermest i tvivl om, og kigger den over skulderen.

`softmax` laver modellens rå point om til sandsynligheder, der summer til 1. Se om du kan
udfylde, hvilken dimension softmax skal arbejde hen over. Cellen finder så det testbillede
med den laveste "mest-sikre" sandsynlighed og tegner både billedet og
sandsynligheds-søjlerne. Hvad er modellen i tvivl imellem?

Kig på formen: `model(X_test)` giver ét point pr. klasse pr. billede — rækker er billeder,
kolonner er de 10 klasser.

Hint: sandsynlighederne for de 10 klasser skal lægge sammen til 1 for HVERT billede — så
softmax skal gå hen over kolonnerne, altså `dim=1`.

In [ ]:
with torch.no_grad():
    probabilities = torch.softmax(model(X_test), dim=...)   # <-- softmax skal summere til 1 hen over de 10 klasser (kolonnerne)

max_probability = probabilities.max(dim=1).values
i = max_probability.argmin().item()          # mest usikre billede

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].imshow(X_test[i].squeeze(), cmap="gray")
axes[0].set_title(f"label: {class_names[y_test[i]]}")
axes[0].axis("off")
axes[1].bar(class_names, probabilities[i])
axes[1].tick_params(axis="x", rotation=45)
plt.show()

##### Opgave 2.8
Vi kigger ind i netværkets øjne: hvad ser hver af de 16 lærte kerner i et billede?

Feature maps-figuren viser præcis det. Cellen kører den allerede på en taske — prøv også en
STØVLE (skift `y_train == 8` ud med `9`). Kan du finde en kerne, der tydeligt er
"kant-agtig" (sammenlign med dine håndbyggede kerner fra afsnit 1)? Og en, der mest ligner
ren støj?

Husk kant-billederne fra opgave 1.4: en kant-kerne fremhæver omridset og gør resten mørkt.
Kig efter feature maps, der ser sådan ud.

Hint: kerner, der lyser op langs tøjets kanter, opfører sig ligesom dine Sobel-kerner.
Kerner, der ser grynede og mønsterløse ud over hele billedet, har netværket ikke fundet
et stærkt mønster til.

In [ ]:
image = X_train[y_train == 8][0]      # taske — prøv også 9 (støvle)

with torch.no_grad():
    fm = model.activation(model.conv1(image.reshape(1, 1, 28, 28)))

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, axis in enumerate(axes.ravel()):
    axis.imshow(fm[0, i], cmap="gray")
    axis.set_title(f"kerne {i}", fontsize=8)
    axis.axis("off")
plt.show()

## Ekstra opgaver

Her er nogle ekstra udfordringer, hvis du er nået hele vejen igennem og har lyst til mere.
De bygger videre på det, du allerede har lavet.

##### Ekstra 1
Nu bygger du din egen kant-kerne — denne gang til DIAGONALE kanter (fra øverst-venstre mod
nederst-højre).

I cellen nedenfor står et forslag med otte af de ni tal på plads (samme idé som Sobel, men
drejet på skrå). Se om du kan udfylde det sidste tal, og test så kernen på sneakeren og på
tasken. Prøv til sidst at finde på mindst ét tjek, der overbeviser dig om, at den virkelig
finder diagonale kanter.

Kig på `vertical_edge` og din `horizontal_edge` fra opgave 1.4: begge har positive tal på
den ene side og negative på den anden. Den diagonale gør det samme, men langs skråen.

Hint: de to modstående hjørner skal have modsat fortegn. Øverste venstre er -2 — hvad skal
nederste højre så være, for at kernen kan mærke forskellen på skrå?

In [ ]:
# En diagonal-kant-kerne reagerer, når billedet skifter langs skråen (øverst-venstre → nederst-højre).
# Otte af de ni tal er fyldt ud — udfyld selv det sidste i nederste højre hjørne:
diagonal_kerne = torch.tensor([[-2., -1.,  0.],
                               [-1.,  0.,  1.],
                               [ 0.,  1., ...]])   # <-- udfyld hjørnet (modsat -2 oppe til venstre)

ud = F.conv2d(shoe.reshape(1, 1, 28, 28), diagonal_kerne.reshape(1, 1, 3, 3))
plt.imshow(ud.squeeze(), cmap="gray")
plt.axis("off")
plt.show()

# prøv den også på en taske: skift shoe ud med X_train[y_train == 8][0]

##### Ekstra 2
**CIFAR-10: rigtige FARVEBILLEDER.** Nu forlader vi gråtoner og prøver et CNN på farvefotos.

Koden nedenfor er komplet: den henter CIFAR-10 (fly, biler, katte... — ~170 MB, men Kaggle
er hurtig), viser billederne og træner et farve-CNN med 3 input-kanaler. Kør den, og
eksperimentér så: flere epoker? flere kerner? Hvor langt kan du komme over de ~50 %?
(Tilfældig gætning er 10 % — men perfekt er UMULIGT på 3 epoker og 4.000 billeder.)

Kig på `ColorNet`: det eneste nye i forhold til `ClothesNet` er `nn.Conv2d(3, ...)` i
stedet for `nn.Conv2d(1, ...)` — tre kanaler (rød, grøn, blå) ind i stedet for én.

Hint: vil du skrue på modellen, så led efter `range(3)` (antal epoker) og de to
`nn.Conv2d`-linjer (antal kerner) — men husk, at `fc`-lagets input skal følge med, hvis du
ændrer kerne-antallet, ligesom i opgave 2.2.

In [ ]:
import torchvision

# CIFAR-10 hentes fra Kaggle (hurtigt!) — torchvision læser de udpakkede filer:
path_cifar = kagglehub.dataset_download("pankrzysiu/cifar10-python")
cifar_train = torchvision.datasets.CIFAR10(root=path_cifar, train=True)
cifar_test = torchvision.datasets.CIFAR10(root=path_cifar, train=False)

#  Plan B (hvis Kaggle driller) — hent direkte fra kilden (kan være LANGSOM):
# cifar_traen = torchvision.datasets.CIFAR10(root="cifar_data", train=True, download=True)
# cifar_test = torchvision.datasets.CIFAR10(root="cifar_data", train=False, download=True)

X_c = torch.tensor(cifar_train.data[:4000] / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
y_c = torch.tensor(cifar_train.targets[:4000], dtype=torch.long)
X_ct = torch.tensor(cifar_test.data[:1000] / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
y_ct = torch.tensor(cifar_test.targets[:1000], dtype=torch.long)
cifar_names = ["fly", "bil", "fugl", "kat", "hjort", "hund", "frø", "hest", "skib", "lastbil"]
print("X_c:", X_c.shape, " ← 3 kanaler: rød, grøn, blå!")

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for i, axis in enumerate(axes):
    axis.imshow(X_c[i].permute(1, 2, 0))       # tilbage til (H, B, kanaler) for imshow
    axis.set_title(cifar_names[y_c[i]])
    axis.axis("off")
plt.show()

class ColorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # 3 kanaler ind!
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.activation = nn.ReLU()
        self.fc = nn.Linear(32 * 8 * 8, 10)                       # 32 → 16 → 8 med padding=1

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
color_model = ColorNet()
optimizer = torch.optim.Adam(color_model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(3):
    for i in range(0, len(X_c), 64):
        optimizer.zero_grad()
        loss = loss_fn(color_model(X_c[i:i + 64]), y_c[i:i + 64])
        loss.backward()
        optimizer.step()
    print(f"epoke {epoch + 1} færdig")

with torch.no_grad():
    acc = (color_model(X_ct).argmax(dim=1) == y_ct).float().mean()
print(f"CIFAR-10 accuracy: {acc.item():.1%}  (tilfældigt = 10 %)")

##### Ekstra 3
**Det ondeste eksperiment.** Nu ødelægger vi med vilje billedernes struktur og ser, hvem
det rammer.

Cellen nedenfor laver en `shuffle`-funktion, der omrokerer alle 784 pixels med den SAMME
tilfældige permutation i hvert billede — så al information er der stadig, men naboskabet er
væk. Det meste er fyldt ud for dig: en lille `train_and_score`-hjælper træner et frisk
netværk og giver dets accuracy tilbage. Se om du kan udfylde den omrokerede testdata, så
begge modeller bliver målt på det samme. Forudsig FØRST: hvem rammes hårdest — og hvorfor?
Kør så og se.

Husk fra opgave 1.9 og 2.6, hvad et CNN's styrke bygger på: at pixels tæt på hinanden hører
sammen.

Hint: dense-nettet ser i forvejen 784 tal uden at vide, hvem der er naboer — så en fast
omrokering generer det knap nok. Hvad sker der til gengæld med et CNN, hvis nabo-pixels ikke
længere ligger ved siden af hinanden?

In [ ]:
perm = torch.randperm(784)

def shuffle(X):
    return X.reshape(-1, 784)[:, perm].reshape(-1, 1, 28, 28)

show_images(shuffle(X_train), y_train, n=5, class_names=class_names)   # "tøj"...

# Vi laver omrokerede udgaver af både trænings- og testdata én gang.
# Samme permutation bruges på begge, så modellerne kan sammenlignes retfærdigt:
X_train_shuf = shuffle(X_train)
X_test_shuf = ...   # <-- udfyld: omrokér testdataen på samme måde som X_train_shuf ovenfor

def train_and_score(net):
    """Træner et frisk netværk på de omrokerede billeder og giver test-accuracy tilbage."""
    torch.manual_seed(42)
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    for epoch in range(5):
        for i in range(0, len(X_train_shuf), 64):
            optimizer.zero_grad()
            loss = loss_fn(net(X_train_shuf[i:i + 64]), y_train[i:i + 64])
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        return (net(X_test_shuf).argmax(dim=1) == y_test).float().mean().item()

# Vi genbruger de to arkitekturer, du allerede kender fra afsnit 2:
cnn_shuffled = train_and_score(ClothesNet())
dense_shuffled = train_and_score(nn.Sequential(nn.Flatten(),
                                               nn.Linear(784, 128), nn.ReLU(),
                                               nn.Linear(128, 10)))
print(f"omrokeret: CNN {cnn_shuffled:.1%} | dense {dense_shuffled:.1%}")